# Notebook 02 — Leakage-Safe Dataset Split

Create train, validation, and test splits without allowing near-duplicate/similar images to cross split boundaries.

The split is based on the `similarity_group_id` created during the dataset quality and leakage analysis.




In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import json

SEED = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "dataset_manifest_with_similarity_groups.csv"
OUTPUT_DIR = PROJECT_ROOT / "dataset_split"

print("Project root:", PROJECT_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Output:", OUTPUT_DIR)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}\n"
        "Run Notebook 01_5_label_quality_leakage_groups.ipynb first."
    )


Project root: d:\driver-drowsiness-detection
Manifest: d:\driver-drowsiness-detection\outputs\dataset_manifest_with_similarity_groups.csv
Output: d:\driver-drowsiness-detection\dataset_split


In [25]:
df = pd.read_csv(MANIFEST_PATH)

print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

display(df.head())


Rows: 2900

Columns:
['path', 'filename', 'class', 'width', 'height', 'channels', 'mode', 'aspect_ratio', 'file_size_kb', 'valid', 'file_md5', 'similarity_group_id']


,path,filename,class,width,height,channels,mode,aspect_ratio,file_size_kb,valid,file_md5,similarity_group_id
0,D:\driver-drowsiness-detection\dataset\train\C...,_0.jpg,Closed,145,145,3,RGB,1.0000,6.09,True,f9319c997f4c75b111bd20bb6e22b1e1,SIM_0001
1,D:\driver-drowsiness-detection\dataset\train\C...,_1.jpg,Closed,142,135,3,RGB,1.0519,10.14,True,5e42467e4a1e07235cc3a0cdf7d32f01,SIM_0002
2,D:\driver-drowsiness-detection\dataset\train\C...,_10.jpg,Closed,131,127,3,RGB,1.0315,25.28,True,fbdbf633838b6268c8fc211258999021,SIM_0003
3,D:\driver-drowsiness-detection\dataset\train\C...,_100.jpg,Closed,288,287,3,RGB,1.0035,17.08,True,435eaa857612d1ae5454501c221abd5d,SIM_0004
4,D:\driver-drowsiness-detection\dataset\train\C...,_101.jpg,Closed,141,123,3,RGB,1.1463,26.18,True,9d0f18f02fc1990be2a0f9edb5376ff5,SIM_0005


## 1. Prepare the manifest


In [26]:
# Check manifest columns
print("Available columns:")
print(df.columns.tolist())

# Detect the image path column automatically
path_candidates = [
    "filepath",
    "file_path",
    "image_path",
    "path",
    "full_path",
    "image_file",
    "filename"
]

path_column = next(
    (col for col in path_candidates if col in df.columns),
    None
)

if path_column is None:
    raise ValueError(
        "Could not identify the image path column. "
        f"Available columns: {df.columns.tolist()}"
    )

# Standardize the path column name for the rest of the notebook
if path_column != "filepath":
    df = df.rename(columns={path_column: "filepath"})

required_cols = {"filepath", "class", "similarity_group_id"}

missing = required_cols - set(df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}\n"
        f"Available columns: {df.columns.tolist()}"
    )

df = df.copy()
df["filepath"] = df["filepath"].astype(str)
df["class"] = df["class"].astype(str)
df["similarity_group_id"] = df["similarity_group_id"].astype(str)

print("\n✓ Manifest structure validated")
print("Image path column:", path_column)
print("Images:", len(df))
print("Similarity groups:", df["similarity_group_id"].nunique())
print("\nClasses:")
print(df["class"].value_counts().sort_index())

Available columns:
['path', 'filename', 'class', 'width', 'height', 'channels', 'mode', 'aspect_ratio', 'file_size_kb', 'valid', 'file_md5', 'similarity_group_id']

✓ Manifest structure validated
Image path column: path
Images: 2900
Similarity groups: 2513

Classes:
class
Closed     726
Open       726
no_yawn    725
yawn       723
Name: count, dtype: int64


## 2. Group-level split




In [27]:
rng = np.random.default_rng(SEED)

group_info = (
    df.groupby("similarity_group_id")
      .agg(
          n_images=("filepath", "size"),
          n_classes=("class", "nunique")
      )
      .reset_index()
)

# Shuffle groups deterministically.
group_info = group_info.sample(frac=1, random_state=SEED).reset_index(drop=True)

total_images = len(df)
target_train = total_images * 0.70
target_val = total_images * 0.15

assignments = {}
train_images = 0
val_images = 0
test_images = 0

for row in group_info.itertuples(index=False):
    gid = row.similarity_group_id
    n = int(row.n_images)

    # Greedy assignment to the split currently furthest below its target.
    train_gap = target_train - train_images
    val_gap = target_val - val_images

    if train_gap >= val_gap and train_gap > 0:
        split = "train"
        train_images += n
    elif val_gap > 0:
        split = "validation"
        val_images += n
    else:
        split = "test"
        test_images += n

    assignments[gid] = split

df["split"] = df["similarity_group_id"].map(assignments)

print("Split image counts:")
print(df["split"].value_counts())

print("\nSplit percentages:")
print((df["split"].value_counts(normalize=True) * 100).round(2))


Split image counts:
split
train         2030
test           435
validation     435
Name: count, dtype: int64

Split percentages:
split
train         70.0
test          15.0
validation    15.0
Name: proportion, dtype: float64


## 3. Verify leakage prevention



In [28]:
group_split_counts = (
    df.groupby("similarity_group_id")["split"]
      .nunique()
)

leaking_groups = group_split_counts[group_split_counts > 1]

print("Groups appearing in multiple splits:", len(leaking_groups))

if len(leaking_groups) > 0:
    display(leaking_groups)
    raise AssertionError("Leakage detected: a similarity group appears in multiple splits.")

print("✓ No similarity group crosses train/validation/test boundaries.")


Groups appearing in multiple splits: 0
✓ No similarity group crosses train/validation/test boundaries.


## 4. Check class distribution



In [29]:
class_split = pd.crosstab(df["split"], df["class"])

print("Class counts by split:")
display(class_split)

print("\nClass percentages within each split:")
class_pct = class_split.div(class_split.sum(axis=1), axis=0) * 100
display(class_pct.round(2))


Class counts by split:


class,Closed,Open,no_yawn,yawn
split,,,,
test,102,112,118,103
train,520,506,500,504
validation,104,108,107,116



Class percentages within each split:


class,Closed,Open,no_yawn,yawn
split,,,,
test,23.45,25.75,27.13,23.68
train,25.62,24.93,24.63,24.83
validation,23.91,24.83,24.60,26.67


## 5. Check source files



In [30]:
missing_files = []

for path in df["filepath"]:
    if not Path(path).exists():
        missing_files.append(path)

print("Missing source files:", len(missing_files))

if missing_files:
    print("\nFirst missing files:")
    for p in missing_files[:10]:
        print(p)
    raise FileNotFoundError("One or more source images are missing.")

print("✓ All source images exist.")


Missing source files: 0
✓ All source images exist.


## 6. Create the split folders



In [31]:
# Create directories
for split in ["train", "validation", "test"]:
    for cls in sorted(df["class"].unique()):
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# Copy files safely
copied = 0
skipped = 0

for _, row in df.iterrows():
    src = Path(row["filepath"])
    split = row["split"]
    cls = row["class"]

    dst = OUTPUT_DIR / split / cls / src.name

    if dst.exists():
        skipped += 1
        continue

    shutil.copy2(src, dst)
    copied += 1

print(f"Newly copied: {copied}")
print(f"Already existed: {skipped}")
print(f"Total represented: {copied + skipped}")
print(f"Output directory: {OUTPUT_DIR}")

Newly copied: 0
Already existed: 2900
Total represented: 2900
Output directory: d:\driver-drowsiness-detection\dataset_split


## 7. Final validation



In [32]:
validation_rows = []

for split in ["train", "validation", "test"]:
    for cls in sorted(df["class"].unique()):
        folder = OUTPUT_DIR / split / cls
        count = len(list(folder.glob("*"))) if folder.exists() else 0
        expected = int(((df["split"] == split) & (df["class"] == cls)).sum())

        validation_rows.append({
            "split": split,
            "class": cls,
            "expected": expected,
            "actual": count,
            "match": expected == count
        })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if not validation_df["match"].all():
    raise AssertionError("Copied file counts do not match the manifest.")

print("✓ Folder counts match the manifest.")


,split,class,expected,actual,match
0,train,Closed,520,520,True
1,train,Open,506,506,True
2,train,no_yawn,500,500,True
3,train,yawn,504,504,True
4,validation,Closed,104,104,True
5,validation,Open,108,108,True
6,validation,no_yawn,107,107,True
7,validation,yawn,116,116,True
8,test,Closed,102,102,True
9,test,Open,112,112,True


✓ Folder counts match the manifest.


In [33]:
# Save the split manifest
split_manifest_path = PROJECT_ROOT / "outputs" / "dataset_split_manifest.csv"
df.to_csv(split_manifest_path, index=False)

# Save summary
summary = {
    "seed": SEED,
    "total_images": int(len(df)),
    "similarity_groups": int(df["similarity_group_id"].nunique()),
    "splits": {
        split: {
            "images": int((df["split"] == split).sum()),
            "percentage": round(float((df["split"] == split).mean() * 100), 2)
        }
        for split in ["train", "validation", "test"]
    },
    "leaking_similarity_groups": int(len(leaking_groups))
}

summary_path = PROJECT_ROOT / "outputs" / "dataset_split_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("Saved:")
print(split_manifest_path)
print(summary_path)


Saved:
d:\driver-drowsiness-detection\outputs\dataset_split_manifest.csv
d:\driver-drowsiness-detection\outputs\dataset_split_summary.json
